**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Hyperparameter Tuning and Model Selection

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 1000
# Imbalanced fraud-like dataset
fraud = np.zeros(n)
fraud[:60] = 1
np.random.shuffle(fraud)

f1 = np.where(fraud == 1, np.random.normal(0.7, 0.2, n), np.random.normal(0.3, 0.2, n)).clip(0, 1)
f2 = np.where(fraud == 1, np.random.normal(0.6, 0.3, n), np.random.normal(0.4, 0.3, n)).clip(0, 1)
f3 = np.random.normal(0.5, 0.2, n).clip(0, 1)

X = pd.DataFrame({'feature_1': f1, 'feature_2': f2, 'feature_3': f3})
y = pd.Series(fraud.astype(int))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Dataset shape: {X.shape}")
print(f"Fraud rate: {y.mean():.1%}")
print("Setup complete.")

## Step 1: Grid Search with Accuracy Scoring

The analyst runs a grid search to find the best hyperparameters for a Random Forest classifier on this imbalanced dataset.

In [ ]:
# AI-generated <- contains Bug 1
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5]
}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',  # <- Bug 1
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.3f}")

**Bug 1 Investigation:** Run the cell above. The best score looks high. With 94% of samples being non-fraud, what score would a model that always predicts non-fraud achieve? Is `scoring='accuracy'` selecting the best model for fraud detection, or the best model at predicting the majority class?

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: Scaling Before Cross-Validation

The analyst scales the features before passing data to a second grid search, to ensure the model receives normalised inputs.

In [ ]:
# AI-generated <- contains Bug 2
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # <- Bug 2: scaler sees all training data at once
X_test_sc = scaler.transform(X_test)

param_grid2 = {'n_estimators': [50, 100], 'max_depth': [3, 5]}
grid_search2 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid2,
    cv=5,
    scoring='roc_auc'
)
grid_search2.fit(X_train_sc, y_train)
print(f"Best params: {grid_search2.best_params_}")
print(f"Best ROC AUC: {grid_search2.best_score_:.3f}")

**Bug 2 Investigation:** Run the cell above. The code runs cleanly. But `scaler.fit_transform(X_train)` is called *before* the CV loop runs — meaning the scaler's mean and standard deviation were computed using data from future validation folds. Why is this a problem? What should the scaler only ever see during cross-validation?

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Cross-Validation Strategy in the Pipeline

The analyst correctly wraps the scaler and model in a Pipeline, but uses the default cross-validation strategy.

In [ ]:
# AI-generated <- contains Bug 3
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])
param_grid3 = {'rf__n_estimators': [50, 100], 'rf__max_depth': [3, 5]}
grid_search3 = GridSearchCV(
    pipeline,
    param_grid3,
    cv=5,  # <- Bug 3: plain KFold, not stratified
    scoring='roc_auc'
)
grid_search3.fit(X_train, y_train)
print(f"Best params: {grid_search3.best_params_}")

**Bug 3 Investigation:** Run the cell above. The pipeline structure looks correct, but `cv=5` uses `KFold` by default — it splits the data sequentially without regard for class balance. With only 6% fraud, what could go wrong in a fold that happens to contain very few or zero fraud samples? What CV class should replace the plain integer?

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below applies all three fixes in a single pipeline. Run it end-to-end to confirm the tuning workflow is now sound.

In [ ]:
# --- Corrected pipeline: all three bugs fixed ---

# Fix 1: use roc_auc scoring — not accuracy — on imbalanced data
# Fix 2: wrap scaler in Pipeline so it is fit only on the CV training split
# Fix 3: use StratifiedKFold to preserve class balance in every fold

pipeline_fixed = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid_fixed = {'rf__n_estimators': [50, 100], 'rf__max_depth': [3, 5]}
skf = StratifiedKFold(n_splits=5)

grid_search_fixed = GridSearchCV(
    pipeline_fixed,
    param_grid_fixed,
    cv=skf,              # Fix 3: stratified folds
    scoring='roc_auc',   # Fix 1: appropriate metric for imbalanced data
    n_jobs=-1
)
grid_search_fixed.fit(X_train, y_train)  # Fix 2: scaler fit inside CV loop via Pipeline

print(f"Best params: {grid_search_fixed.best_params_}")
print(f"Best CV ROC AUC: {grid_search_fixed.best_score_:.3f}")

best_model = grid_search_fixed.best_estimator_
y_proba = best_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba)
print(f"Test ROC AUC: {test_auc:.3f}")